# 11. SQLite Effects: 롤백, 재시도, 오래된 완료의 경쟁

원본: https://github.com/deepseek-ai/dsh-superagent-prototype

근거 파일: `src/app/effects.ts`, `outbox.ts`, `store.ts`, `src/sources/github-channel.ts`; `tests/effects.spec.ts`

**범위:** 원본은 TypeScript/Cordis입니다. 아래는 원본의 일부 불변조건을 Python/LangGraph로 실험하며 공식 API 호환 구현이 아닙니다. 새 커널에서 순서대로 실행합니다. 외부 API 호출이 없습니다.

## 왜 graph checkpoint만으로 부족한가
승인 상태 저장과 메시지 전송 사이에 장애가 나면 무엇이 남을까요? 원본은 도메인 변경과 effect enqueue를 같은 SQLite 트랜잭션에 넣습니다. 외부 I/O는 커밋 후 수행합니다. Effects는 `(kind,key)`로 합치고 payload 갱신 시 generation을 증가시킵니다. 예전 실행의 완료가 새 payload를 삭제하지 않도록 seq/generation을 비교합니다.

아래는 실패를 의도적으로 주입하는 실습입니다. SQLite 및 원본과 같은 조건부 DELETE를 실제 실행하지만 production queue 전체를 구현하지 않습니다.

In [1]:
import sqlite3, tempfile
from pathlib import Path
from contextlib import closing
workspace=tempfile.TemporaryDirectory()
path=Path(workspace.name)/'effects.sqlite'
db=sqlite3.connect(path)
db.executescript('CREATE TABLE tickets(id TEXT PRIMARY KEY); CREATE TABLE effect(kind TEXT,key TEXT,payload TEXT,generation INTEGER,PRIMARY KEY(kind,key));')
def enqueue(c,key,payload):
    c.execute("INSERT INTO effect VALUES ('reply',?,?,1) ON CONFLICT(kind,key) DO UPDATE SET payload=excluded.payload,generation=effect.generation+1",(key,payload))
try:
    with db:
        db.execute('INSERT INTO tickets VALUES (?)',('rollback',))
        enqueue(db,'r1','발송하면 안 되는 메시지')
        raise RuntimeError('커밋 직전 장애')
except RuntimeError: pass
assert db.execute('SELECT count(*) FROM tickets').fetchone()[0]==0
assert db.execute('SELECT count(*) FROM effect').fetchone()[0]==0
with db:
    db.execute('INSERT INTO tickets VALUES (?)',('t1',))
    enqueue(db,'r1','version 1')
old=db.execute('SELECT generation FROM effect WHERE key=?',('r1',)).fetchone()[0]
with db: enqueue(db,'r1','version 2')
with db:
    deleted=db.execute('DELETE FROM effect WHERE key=? AND generation=?',('r1',old)).rowcount
assert deleted==0
assert db.execute('SELECT payload FROM effect').fetchone()[0]=='version 2'
db.close()
with closing(sqlite3.connect(path)) as fresh:
    assert fresh.execute('SELECT payload FROM effect').fetchone()[0]=='version 2'
print('롤백 원자성, 오래된 완료 거부, DB 재연결 후 잔여 effect 보존 확인')
workspace.cleanup()

롤백 원자성, 오래된 완료 거부, DB 재연결 후 잔여 effect 보존 확인


## 전송 성공 후 receipt 기록 전에 장애가 나면?
로컬 outbox의 중복 key만으로 외부 전송의 exactly-once를 보장할 수 없습니다. 원본 GitHub adapter는 `<!-- dsh-superagent:메시지ID -->` 마커를 기존 댓글에서 찾고, 있으면 재전송하지 않습니다. Feishu도 provider idempotency를 이용합니다. GitHub 방식은 read-before-write이므로 외부 서버의 원자적 멱등키와 동일하지 않고, 동시 writer·조회 가시성 등의 가정을 검토해야 합니다.

In [2]:
external=[]
def send_with_marker(message_id,text):
    marker=f'<!-- dsh-superagent:{message_id} -->'
    if any(marker in body for body in external): return 'already-sent'
    external.append(text+'\n'+marker)
    return 'sent'
# 외부에는 전달됐지만 로컬 receipt 기록 전에 장애가 났다고 가정
assert send_with_marker('m1','결과')=='sent'
assert send_with_marker('m1','결과')=='already-sent'
assert len(external)==1
print('로컬 재시도 시 기존 외부 마커 확인:',len(external),'message')

로컬 재시도 시 기존 외부 마커 확인: 1 message


## 과제와 해설
과제: 두 sender가 동시에 조회하면 어떻게 될지 순서도를 그리세요. 해설: A 조회(없음) → B 조회(없음) → A POST → B POST이면 중복 가능합니다. 원본 stateDir lock과 reply concurrency=1은 단일 프로세스 내 경쟁을 제한하지만 외부 서비스 전체의 exactly-once 증명이 아닙니다. 다음 실습으로 retry backoff, pause, dropped receipt, generation 재등록을 추가해 보세요.